# colab_clue_rl

> Phase 12, Track B on a GPU (`specs/multiagent/phase12/`): the warm start and the RL runs of a 3B open-weight model on the Phase 11 clue board, as a standalone Google Colab notebook (an A100 runtime; any Linux GPU machine works the same way).

[Open in Colab](https://colab.research.google.com/github/Positron00/explRL/blob/main/nbs/colab_clue_rl.ipynb). Outputs go to Google Drive (`MyDrive/explRL_rl/`); nothing here needs an API key or the repo's `results/`. Run the sections in order, one session each where noted; every section resumes after a disconnection.

1. **Set-up**: GPU check, install, Drive.
2. **Pilot** (about 1 hour): a few warm-start and RL steps; step time and memory, so that each main run fits a few hours.
3. **Warm start** (about 1 hour): supervised training on the rule agent's moves; the adapter is checked on the Mac (`demo_rl`).
4. **RL run** (1.5-2 hours, one session per reward scheme): set `SCHEME`, run; needs `RL_PLAN` frozen.
5. **Pack**: a zip of the adapters, metrics and configurations to download to `results/multiagent/rl/colab/` on the Mac.

The sections are behind the `gpu` test flag, so normal tests only run the configuration cell.

In [ ]:
#| gpu
import subprocess, sys
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'explRL[train] @ git+https://github.com/Positron00/explRL'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/explRL_rl'
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU: use a GPU runtime')

In [ ]:
import json
import time
from pathlib import Path

from explRL.coop_grid import MODELS, RL_PLAN, coopHash
from explRL.train import demo_artifact_dir

OUT = Path(globals().get('OUT') or demo_artifact_dir('colab', 'multiagent/rl'))
BASE = MODELS['base']
SCHEME = 'credit'                      # set before each RL session: 'individual', 'team' or 'credit'
SFT_SEEDS = range(RL_PLAN['sftData']['seeds'][0], RL_PLAN['sftData']['seeds'][1] + 1)
print(f"RL_PLAN {coopHash(RL_PLAN)} (frozen: {RL_PLAN['frozen']}); base model {BASE}; outputs {OUT}")

## Pilot (`gpu`): step time and memory

In [ ]:
#| gpu
from explRL.clue_rl import RLConfig, trainRL
from explRL.clue_sft import SFTConfig, mergeAdapter, sftExamples, trainSFT
P = OUT / 'pilot'
t0 = time.time()
adapter = trainSFT(SFTConfig(BASE, str(P / 'sft'), maxSteps=20), sftExamples(range(20000, 20010), turns=20))
t1 = time.time()
mergeAdapter(BASE, adapter, P / 'warm')
trainRL(RLConfig(str(P / 'warm'), str(P / 'rl'), scheme=SCHEME, **{**RL_PLAN['rl'], 'steps': 3, 'ckptEvery': 1}))
steps = [json.loads(x) for x in open(P / 'rl' / 'metrics.jsonl')]
per = sum(x['rolloutSeconds'] + x['updateSeconds'] for x in steps) / len(steps)
print(f"warm start: {(t1 - t0) / 20:.1f} s per step; RL: {per:.0f} s per step -> {per * RL_PLAN['rl']['steps'] / 3600:.1f} h "
      f"per run; peak GPU memory {torch.cuda.max_memory_allocated() / 1e9:.1f} GB; session {(time.time() - t0) / 3600:.2f} h")

## Warm start (`gpu`)

In [ ]:
#| gpu
from explRL.clue_sft import SFTConfig, sftExamples, trainSFT
t0 = time.time()
ex = sftExamples(SFT_SEEDS, sizes=tuple(RL_PLAN['sftData']['sizes']), turns=RL_PLAN['sftData']['turns'])
adapter = trainSFT(SFTConfig(BASE, str(OUT / 'warm'), **RL_PLAN['sft']), ex)
print(f'{len(ex)} examples; adapter {adapter}; {(time.time() - t0) / 3600:.2f} GPU hours. Next: download it and run the '
      'warm-start check on the Mac (demo_rl).')

## RL run (`gpu`; one session per reward scheme)

In [ ]:
#| gpu
from explRL.clue_rl import RLConfig, trainRL
from explRL.clue_sft import mergeAdapter
assert RL_PLAN['frozen'], 'freeze and commit RL_PLAN (after the pilot and the warm-start check) before the main runs'
WARM = OUT / 'warm_model'
if not (WARM / 'config.json').exists():
    mergeAdapter(BASE, OUT / 'warm' / 'adapter', WARM)
t0 = time.time()
final = trainRL(RLConfig(str(WARM), str(OUT / 'runs' / SCHEME), scheme=SCHEME, **RL_PLAN['rl']))
print(f'{SCHEME}: {final}; this session {(time.time() - t0) / 3600:.2f} GPU hours')

## Pack (`gpu`): adapters, metrics and configurations for the Mac

In [ ]:
#| gpu
import zipfile
with zipfile.ZipFile(OUT / 'explRL_rl.zip', 'w') as z:
    for p in sorted(OUT.rglob('*')):
        rel = p.relative_to(OUT)
        if p.is_file() and rel.parts[0] in ('warm', 'runs', 'pilot') and 'ckpt-' not in str(rel) and rel.parts[:2] != ('pilot', 'warm'):
            z.write(p, rel)
print(OUT / 'explRL_rl.zip', round((OUT / 'explRL_rl.zip').stat().st_size / 1e6, 1), 'MB')